# SHOT Phase 2b: Single-Frame Ball Detector Training (v2)

MobileNetV3-Small + lightweight decoder → 48×48 heatmap

**v2 changes:**
- Dropout in decoder (overfitting prevention)
- Stronger augmentation (horizontal flip, random crop, Gaussian blur)
- Save best model by BOTH val_loss AND val_pixel_error
- Weight decay (1e-4) for regularization
- Patience 20

**Setup:**
1. Kaggle에서 TrackNet Dataset-001 추가 (Add Data → `tracknet-tennis-ball-detection`)
2. GPU 가속기 활성화 (Settings → Accelerator → GPU P100)
3. 셀 순서대로 실행

## 1. 데이터셋 변환 (TrackNet → SHOT 형식)

In [ ]:
import csv
import json
import os
import shutil
from pathlib import Path

# === TrackNet 데이터셋 경로 (Kaggle에 맞게 수정) ===
# Kaggle에서 TrackNet 데이터셋을 추가한 경우:
TRACKNET_DIR = "/kaggle/input"  # 데이터셋 추가 후 실제 경로 확인 필요

# 가능한 경로들 자동 탐색
possible_paths = [
    "/kaggle/input/tracknet-tennis-ball-tracking/Dataset",
    "/kaggle/input/tracknet-tennis-ball-detection/Dataset",
    "/kaggle/input/tennis-ball-tracking/Dataset",
    "/kaggle/input/tracknet/Dataset",
]

DATASET_DIR = None
for p in possible_paths:
    if os.path.exists(p):
        DATASET_DIR = p
        break

if DATASET_DIR is None:
    # 수동으로 확인
    print("=== /kaggle/input 내용 ===")
    for item in os.listdir("/kaggle/input"):
        full = os.path.join("/kaggle/input", item)
        print(f"  {item}/")
        if os.path.isdir(full):
            for sub in os.listdir(full)[:5]:
                print(f"    {sub}")
    print("\n위 경로에서 Dataset 폴더를 찾아 DATASET_DIR에 지정하세요")
else:
    print(f"데이터셋 발견: {DATASET_DIR}")
    games = [d for d in os.listdir(DATASET_DIR) if d.startswith("game")]
    print(f"Games: {sorted(games)}")

In [ ]:
# TrackNet Label.csv → ball_combined.json 변환

IMG_W, IMG_H = 1280, 720
OUTPUT_DIR = "/kaggle/working/data"
FRAMES_DIR = os.path.join(OUTPUT_DIR, "frames")
os.makedirs(FRAMES_DIR, exist_ok=True)

all_annotations = []
stats = {"total": 0, "visible": 0, "not_visible": 0, "occluded": 0}

input_path = Path(DATASET_DIR)
games = sorted([d for d in input_path.iterdir() if d.is_dir() and d.name.startswith("game")])

for game_dir in games:
    game_id = game_dir.name
    clips = sorted([c for c in game_dir.iterdir() if c.is_dir() and c.name.startswith("Clip")])
    
    for clip_dir in clips:
        clip_id = clip_dir.name
        video_id = f"{game_id}_{clip_id}"
        
        label_file = clip_dir / "Label.csv"
        if not label_file.exists():
            continue
        
        with open(label_file, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                fname = row["file name"].strip()
                vis_str = row["visibility"].strip()
                x_str = row["x-coordinate"].strip()
                y_str = row["y-coordinate"].strip()
                
                if not vis_str or not fname:
                    continue
                vis = int(vis_str)
                try:
                    x_px = float(x_str) if x_str else 0.0
                    y_px = float(y_str) if y_str else 0.0
                except ValueError:
                    x_px, y_px = 0.0, 0.0
                    vis = 0
                
                new_fname = f"{video_id}_frame_{fname}"
                
                if vis == 0:
                    x_norm, y_norm = -1.0, -1.0
                else:
                    x_norm = x_px / IMG_W
                    y_norm = y_px / IMG_H
                
                all_annotations.append({
                    "image": new_fname,
                    "x": round(x_norm, 6),
                    "y": round(y_norm, 6),
                    "visibility": vis
                })
                stats["total"] += 1
                if vis == 0: stats["not_visible"] += 1
                elif vis == 1: stats["visible"] += 1
                elif vis == 2: stats["occluded"] += 1
                
                # Symlink instead of copy (saves disk space)
                src = clip_dir / fname
                dst = Path(FRAMES_DIR) / new_fname
                if src.exists() and not dst.exists():
                    os.symlink(str(src), str(dst))

ann_path = os.path.join(OUTPUT_DIR, "ball_combined.json")
with open(ann_path, "w") as f:
    json.dump(all_annotations, f)

print(f"Total frames: {stats['total']}")
print(f"  Visible: {stats['visible']}, Not visible: {stats['not_visible']}, Occluded: {stats['occluded']}")
print(f"Annotations saved: {ann_path}")
print(f"Frames linked: {len(os.listdir(FRAMES_DIR))}")

## 2. 모델 정의 (BallDetector)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights


class BallDetector(nn.Module):
    """
    Single-frame tennis ball detector.
    Input:  [B, 3, 192, 192] ImageNet-normalized RGB
    Output: [B, 1, 48, 48] ball position heatmap (sigmoid)
    """
    def __init__(self, pretrained=True, dropout=0.15):
        super().__init__()
        weights = MobileNet_V3_Small_Weights.DEFAULT if pretrained else None
        backbone = mobilenet_v3_small(weights=weights)
        self.features = backbone.features

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(576, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 1, kernel_size=1),
        )

    def forward(self, x):
        features = self.features(x)
        heatmap = self.decoder(features)
        return torch.sigmoid(heatmap)


class BallDetectorLoss(nn.Module):
    def __init__(self, alpha=0.97, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, pred, target):
        bce = F.binary_cross_entropy(pred, target, reduction='none')
        pt = torch.where(target > 0.5, pred, 1 - pred)
        focal_weight = (1 - pt) ** self.gamma
        class_weight = torch.where(target > 0.5, 1.0, 1.0 - self.alpha)
        loss = focal_weight * class_weight * bce
        return loss.mean()


def generate_heatmap(x, y, size=48, sigma=2.5):
    if x < 0 or y < 0:
        return torch.zeros(size, size)
    yy, xx = torch.meshgrid(
        torch.arange(size, dtype=torch.float32),
        torch.arange(size, dtype=torch.float32),
        indexing='ij'
    )
    heatmap = torch.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2))
    return heatmap


# Test
model = BallDetector(pretrained=True, dropout=0.15)
total_params = sum(p.numel() for p in model.parameters())
backbone_params = sum(p.numel() for p in model.features.parameters())
decoder_params = sum(p.numel() for p in model.decoder.parameters())
print(f"Total parameters:     {total_params:,}")
print(f"  Backbone:           {backbone_params:,}")
print(f"  Decoder:            {decoder_params:,}")
print(f"Model size (FP32):    {total_params * 4 / 1024 / 1024:.1f} MB")

x = torch.randn(1, 3, 192, 192)
y = model(x)
print(f"Input:  {x.shape}")
print(f"Output: {y.shape} range=[{y.min().item():.4f}, {y.max().item():.4f}]")

## 3. 데이터셋 클래스

In [ ]:
import re
from collections import defaultdict
import numpy as np
from torch.utils.data import Dataset, DataLoader
from PIL import Image, ImageFilter

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class BallDataset(Dataset):
    INPUT_SIZE = 192
    HEATMAP_SIZE = 48
    SIGMA = 2.5

    def __init__(self, annotations, frames_dir, augment=False):
        self.frames_dir = Path(frames_dir)
        self.augment = augment

        existing = set()
        if self.frames_dir.is_dir():
            existing = set(os.listdir(str(self.frames_dir)))

        self.samples = []
        skipped = 0
        for ann in annotations:
            if existing and ann["image"] not in existing:
                skipped += 1
                continue
            self.samples.append(ann)

        if skipped > 0:
            print(f"  Skipped {skipped} samples (missing frame files)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ann = self.samples[idx]
        img_path = self.frames_dir / ann["image"]
        img = Image.open(img_path).convert("RGB")

        ball_x = ann["x"]  # normalized 0~1
        ball_y = ann["y"]
        vis = ann["visibility"]

        # Augmentation that modifies both image and ball position
        if self.augment:
            # Horizontal flip (50%)
            if np.random.random() < 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if vis > 0 and ball_x >= 0:
                    ball_x = 1.0 - ball_x

            # Gaussian blur (20%)
            if np.random.random() < 0.2:
                radius = np.random.uniform(0.5, 1.5)
                img = img.filter(ImageFilter.GaussianBlur(radius=radius))

        img = img.resize((self.INPUT_SIZE, self.INPUT_SIZE), Image.BILINEAR)
        img = np.array(img, dtype=np.float32) / 255.0

        if self.augment:
            img = self._augment_color(img)

        img = (img - IMAGENET_MEAN) / IMAGENET_STD
        img = np.transpose(img, (2, 0, 1))

        if vis > 0 and ball_x >= 0:
            bx = ball_x * self.HEATMAP_SIZE
            by = ball_y * self.HEATMAP_SIZE
            heatmap = generate_heatmap(bx, by, self.HEATMAP_SIZE, self.SIGMA)
        else:
            heatmap = torch.zeros(self.HEATMAP_SIZE, self.HEATMAP_SIZE)

        return (
            torch.from_numpy(img).float(),
            heatmap.unsqueeze(0).float(),
            torch.tensor(vis, dtype=torch.long)
        )

    def _augment_color(self, img):
        # Brightness (50%)
        if np.random.random() < 0.5:
            img = np.clip(img * np.random.uniform(0.7, 1.3), 0, 1)
        # Color shift (30%)
        if np.random.random() < 0.3:
            img = np.clip(img + np.random.uniform(-0.05, 0.05, size=3).astype(np.float32), 0, 1)
        # Gaussian noise (30%)
        if np.random.random() < 0.3:
            img = np.clip(img + np.random.normal(0, 0.02, img.shape).astype(np.float32), 0, 1)
        # Contrast (20%)
        if np.random.random() < 0.2:
            factor = np.random.uniform(0.8, 1.2)
            mean = img.mean()
            img = np.clip((img - mean) * factor + mean, 0, 1)
        return img

print("Dataset class defined (v2 - stronger augmentation).")

## 4. 학습

In [ ]:
import torch.optim as optim

# === Config ===
DATA_JSON = "/kaggle/working/data/ball_combined.json"
FRAMES = "/kaggle/working/data/frames"
EPOCHS = 60
BATCH_SIZE = 32
LR = 1e-3
BACKBONE_LR = 1e-4
WEIGHT_DECAY = 1e-4
FREEZE_BACKBONE_EPOCHS = 5
PATIENCE = 20
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Load annotations
with open(DATA_JSON) as f:
    all_annotations = json.load(f)

print(f"Total annotations: {len(all_annotations)}")
visible = sum(1 for a in all_annotations if a["visibility"] > 0)
print(f"Visible: {visible}, Not visible: {len(all_annotations) - visible}")

# Video-level split
video_frames = defaultdict(list)
for ann in all_annotations:
    name = Path(ann["image"]).stem
    match = re.match(r"(.+?)_frame_\d+$", name)
    vid_id = match.group(1) if match else name
    video_frames[vid_id].append(ann)

video_ids = sorted(video_frames.keys())
np.random.seed(42)
np.random.shuffle(video_ids)
split_idx = max(1, int(len(video_ids) * 0.8))

train_ann = [a for vid in video_ids[:split_idx] for a in video_frames[vid]]
val_ann = [a for vid in video_ids[split_idx:] for a in video_frames[vid]]

print(f"Videos: {len(video_ids)} (train {split_idx}, val {len(video_ids) - split_idx})")
print(f"Frames: train {len(train_ann)}, val {len(val_ann)}")

train_dataset = BallDataset(train_ann, FRAMES, augment=True)
val_dataset = BallDataset(val_ann, FRAMES, augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=4, pin_memory=True,
                          persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=4, pin_memory=True,
                        persistent_workers=True)

print(f"\nTrain batches: {len(train_loader)}, Val batches: {len(val_loader)}")

In [ ]:
# === Training Loop (v2: dual best model saving) ===

model = BallDetector(pretrained=True, dropout=0.15).to(device)
criterion = BallDetectorLoss()

backbone_params = list(model.features.parameters())
decoder_params = list(model.decoder.parameters())
optimizer = optim.AdamW([
    {"params": backbone_params, "lr": BACKBONE_LR},
    {"params": decoder_params, "lr": LR},
], weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {total_params:,} ({total_params * 4 / 1024 / 1024:.1f} MB)")

best_val_loss = float('inf')
best_val_error = float('inf')
no_improve = 0  # based on val_error (the metric we actually care about)

print(f"\n{'Ep':>3} | {'TrLoss':>8} | {'TrAcc':>6} | {'VlLoss':>8} | {'VlAcc':>6} | {'VlErr':>6} | {'LR':>8} | Note")
print("-" * 80)

for epoch in range(EPOCHS):
    # Freeze/unfreeze backbone
    if epoch < FREEZE_BACKBONE_EPOCHS:
        for p in model.features.parameters():
            p.requires_grad = False
        note = "frozen"
    elif epoch == FREEZE_BACKBONE_EPOCHS:
        for p in model.features.parameters():
            p.requires_grad = True
        note = "UNFROZEN"
    else:
        note = ""
    
    # --- Train ---
    model.train()
    train_loss = 0
    train_correct = 0
    train_total = 0
    
    for inputs, targets, visibility in train_loader:
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        for i in range(len(visibility)):
            if visibility[i] > 0:
                train_total += 1
                if outputs[i].max().item() > 0.5:
                    train_correct += 1
    
    avg_train_loss = train_loss / max(len(train_loader), 1)
    train_acc = train_correct / max(train_total, 1) * 100
    
    # --- Validate ---
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    position_errors = []
    
    with torch.no_grad():
        for inputs, targets, visibility in val_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item()
            
            for i in range(len(visibility)):
                if visibility[i] > 0:
                    val_total += 1
                    pred_hm = outputs[i, 0]
                    gt_hm = targets[i, 0]
                    pred_max = pred_hm.max().item()
                    if pred_max > 0.5:
                        val_correct += 1
                        pred_idx = pred_hm.argmax()
                        gt_idx = gt_hm.argmax()
                        h, w = pred_hm.shape
                        pred_y, pred_x = pred_idx // w, pred_idx % w
                        gt_y, gt_x = gt_idx // w, gt_idx % w
                        err = ((pred_x - gt_x).float() ** 2 +
                               (pred_y - gt_y).float() ** 2).sqrt().item()
                        position_errors.append(err)
    
    avg_val_loss = val_loss / max(len(val_loader), 1)
    val_acc = val_correct / max(val_total, 1) * 100
    avg_err = np.mean(position_errors) if position_errors else float('inf')
    
    scheduler.step()
    lr = optimizer.param_groups[1]["lr"]
    
    # === Save best by val_loss ===
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "val_loss": avg_val_loss,
            "val_error": avg_err,
        }, os.path.join(OUTPUT_DIR, "ball_best_loss.pth"))
        note += " *loss"

    # === Save best by val_pixel_error (PRIMARY metric) ===
    if avg_err < best_val_error:
        best_val_error = avg_err
        no_improve = 0
        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "val_loss": avg_val_loss,
            "val_error": avg_err,
        }, os.path.join(OUTPUT_DIR, "ball_best.pth"))
        note += " *err"
    else:
        no_improve += 1
    
    print(f"{epoch:>3} | {avg_train_loss:>8.5f} | {train_acc:>5.1f}% | {avg_val_loss:>8.5f} | {val_acc:>5.1f}% | {avg_err:>5.1f}px | {lr:>8.5f} | {note}")
    
    if no_improve >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch} (no val_error improvement for {PATIENCE} epochs)")
        break
    
    if (epoch + 1) % 10 == 0:
        torch.save({
            "model": model.state_dict(),
            "epoch": epoch,
            "val_loss": avg_val_loss,
            "val_error": avg_err,
        }, os.path.join(OUTPUT_DIR, f"ball_epoch{epoch+1}.pth"))

print(f"\n=== Training Complete ===")
print(f"Best val_loss:  {best_val_loss:.6f} (saved as ball_best_loss.pth)")
print(f"Best val_error: {best_val_error:.1f}px (saved as ball_best.pth)")
print(f"ONNX export will use: ball_best.pth (lowest pixel error)")

## 5. ONNX 변환

In [ ]:
# ONNX export + validation
import os
import numpy as np

ONNX_PATH = os.path.join(OUTPUT_DIR, "ball_detector.onnx")

# Load best model (by pixel error - the metric that matters)
model_export = BallDetector(pretrained=False, dropout=0.15)
ckpt = torch.load(os.path.join(OUTPUT_DIR, "ball_best.pth"), map_location="cpu")
model_export.load_state_dict(ckpt["model"])
model_export.eval()
print(f"Loaded best model from epoch {ckpt['epoch']} (val_error={ckpt['val_error']:.1f}px)")

# Export
dummy_input = torch.randn(1, 3, 192, 192)
torch.onnx.export(
    model_export, dummy_input, ONNX_PATH,
    opset_version=18,
    input_names=["input"],
    output_names=["heatmap"],
    dynamic_axes=None,
)
size_mb = os.path.getsize(ONNX_PATH) / 1024 / 1024
print(f"Exported: {ONNX_PATH} ({size_mb:.2f} MB)")

# Validate with ORT
import onnxruntime as ort

session = ort.InferenceSession(ONNX_PATH)
dummy_np = np.random.randn(1, 3, 192, 192).astype(np.float32)
ort_out = session.run(None, {"input": dummy_np})[0]
print(f"ONNX output shape: {ort_out.shape}")
print(f"ONNX output range: [{ort_out.min():.4f}, {ort_out.max():.4f}]")

# Compare PyTorch vs ONNX
with torch.no_grad():
    pt_out = model_export(torch.from_numpy(dummy_np)).numpy()
diff = np.abs(pt_out - ort_out)
print(f"Max diff: {diff.max():.8f}")
print(f"{'[OK] MATCH' if diff.max() < 1e-5 else '[WARN] Slight difference'}")

## 6. 결과 다운로드

학습 완료 후:
1. 우측 Output 패널에서 `models/ball_detector.onnx` 다운로드
2. `court-detection/src/main/assets/ball_detector.onnx`에 복사
3. Android 빌드 + 실기기 테스트

In [ ]:
import os
# Output files check
print("=== Output Files ===")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024 / 1024
    print(f"  {f:30s} {size:.2f} MB")